# 02 - Ingeniería de Características

## Descripción General

La ingeniería de características (*feature Engineering*) es el proceso de transformar datos brutos en entradas significativas (características) que ayuden a los modelos de machine learning a aprender patrones de manera más efectiva y mejorar la precisión de las predicciones. Esto implica la creación de nuevas características, transformación de las características existentes, imputación de valores faltantes, codificación de datos categóricos y selección de las características más relevantes.

Dado que el rendimiento de un modelo depende en gran medida de la calidad de los datos de entrenamiento, la ingeniería de características se convierte en un paso crucial del preprocesamiento. Su objetivo es transformar y seleccionar los aspectos más relevantes de los datos brutos, alineándolos tanto con la tarea predictiva como con las necesidades específicas del modelo, para maximizar su capacidad de aprendizaje y generalización.

En esta etapa dejamos de “explorar” y empezamos a construir lo que el modelo realmente va a usar. Todo el EDA previo tuvo un propósito claro: identificar qué variables contienen señal y cuáles no. Aquí convertiremos ese análisis en un proceso reproducible.

Partiremos del conjunto de datos `df_features` que contiene las características candidatas al modelo y construiremos un conjunto de variables (X) que capturen las señales mas relevantes sobre los factores que influyen en la fijación de precios de los alojamientos de Airbnb en la Ciudad de México. Para lograrlo, se dividirá el proceso en dos niveles:

- **Creación de features (`build_features`):**  
  Generar nuevas variables a partir de los datos originales. Estas pueden incluir features o métricas derivadas, combinaciones de atributos o indicadores que aporten señales adicionales al modelo.

- **Transformación de features (`transform_features`):**  
  Modificar las variables existentes para optimizar su representación. Esto abarca operaciones como normalización, escalado, codificación de categorías o transformaciones matemáticas que mejoren la capacidad predictiva.

**Flujo completo**


```text
df_clean
   ↓
df_features
   ↓
build_features()
   ↓
transform_features()
   ↓
df_model → X listo para modelado

## Título 2

### Importación de librerías y carga del dataset

In [1]:
# Import libraries, modules and dataset
import pandas as pd
import numpy as np
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2 

# Import eda modules
from src.features.validate_feature import validate_feature
from src.eda.visualization import plot_barplot, plot_boxplot, plot_histogram, plot_regplot


# Load dataset
df_features = pd.read_csv("../data/processed/df_features.csv")

# Preview first rows
df_features.head()

,listing_url,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,number_of_reviews,price,clean_price,log_price,price_per_guest
0,https://www.airbnb.com/rooms/35797,"Mexico City, D.f., Mexico",Cuajimalpa de Morelos,NaN,19.38283,-99.27178,Entire villa,Entire home/apt,2,1.0,...,NaN,NaN,NaN,NaN,NaN,0,"$3,673.00",3673.0,8.208764,1836.500000
1,https://www.airbnb.com/rooms/44616,NaN,Cuauhtémoc,NaN,19.41162,-99.17794,Entire home,Entire home/apt,14,5.5,...,4.70,4.87,4.78,4.98,4.47,65,"$18,000.00",18000.0,9.798127,1285.714286
2,https://www.airbnb.com/rooms/56074,"Mexico City, DF, Mexico",Cuauhtémoc,NaN,19.43977,-99.15605,Entire condo,Entire home/apt,2,1.0,...,4.88,4.98,4.94,4.76,4.79,84,$591.00,591.0,6.381816,295.500000
3,https://www.airbnb.com/rooms/165772,"Mexico City, Federal District, Mexico",Miguel Hidalgo,NaN,19.40826,-99.18659,Entire home,Entire home/apt,16,5.0,...,4.84,4.91,4.89,4.75,4.90,386,"$3,673.00",3673.0,8.208764,229.562500
4,https://www.airbnb.com/rooms/171109,"Mexico City, Federal District, Mexico",Benito Juárez,NaN,19.39675,-99.17581,Private room in rental unit,Private room,2,1.0,...,4.61,4.98,4.95,4.97,4.83,123,$321.00,321.0,5.771441,160.500000


## Creación de Características (Build Features)

En esta sección se construirán las variables derivadas que constituirán el conjunto de entrada (X) para el modelo. A partir de las variables originales, se generan nuevas features con capacidad explicativa, incorporando relaciones, agregaciones y transformaciones identificadas durante el EDA.

El objetivo no es aumentar el número de variables, sino capturar mejor la señal relevante: ubicación, características del listing, calidad percibida y contexto del entorno. Estas features se desarrollaran de forma modular para facilitar su validación, reutilización y posterior integración en el pipeline de modelado.

## Features de ubicación (Geo)

La ubicación es uno de los factores más determinantes en el precio de un listing. Sin embargo, variables como latitud y longitud por sí solas no son directamente interpretables por el modelo. En esta sección se construirán features geoespaciales que capturan accesibilidad y atractivo del entorno, como la cercanía a zonas turísticas o culturales y la densidad de actividad comercial en la zona. Con estas variables buscaremos traducir la ubicación en señales más informativas y comparables entre listings.

In [4]:
# Import the function to add geo features
from src.features.build_features import add_distance_to_attractions
from src.features.build_features import add_attractions_density
from src.features.build_features import add_commercial_density

# Apply the function to df_features to add the new columns
df_features = add_distance_to_attractions(df_features)
df_features = add_attractions_density(df_features)
df_features = add_commercial_density(df_features)

# Display the first rows of latitude, longitude, and the new geo features
geo_features = ['dist_to_nearest_attraction', 'attractions_within_radius', 'commercial_within_radius']
df_features[["latitude", "longitude"] + geo_features]


,latitude,longitude,dist_to_nearest_attraction,attractions_within_radius,commercial_within_radius
0,19.382830,-99.271780,2.450990,0,1
1,19.411620,-99.177940,0.415358,4,252
2,19.439770,-99.156050,0.375601,3,129
3,19.408260,-99.186590,0.985700,1,53
4,19.396750,-99.175810,1.557305,0,116
...,...,...,...,...,...
21511,19.442240,-99.113440,2.178589,0,14
21512,19.308017,-99.168158,0.958782,1,29
21513,19.434460,-99.174010,0.832345,1,128
21514,19.406435,-99.160934,0.958943,1,184


In [5]:
# Validate dist_to_nearest_attraction
validate_feature(df_features, geo_features, 'log_price')


========== Validating: dist_to_nearest_attraction ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0   float64  21516     21516   18100     0           0.0

--- Statistical Summary ---
                          0
mean               1.302159
median             0.716779
mode               0.606966
std                1.588216
min                0.004748
p25                0.395287
p50                0.716779
p75                1.571468
max               16.244343
skew               2.974835
kurt              11.991718
outliers_count  1839.000000
outliers_pct       8.550000
pearson_r         -0.238944

========== Validating: attractions_within_radius ==========

--- Basic Profile ---
  data_type  count  non_null  unique  null  null_ratio_%
0     int64  21516     21516      12     0           0.0

--- Statistical Summary ---
                        0
mean             2.717048
median           2.000000
mode             0.000000
std              2.79